In [1]:
import polars as pl
import pandas as pd
from pathlib import Path
from glob import glob

DATA = Path("../data")
OUTPUTS = Path("../outputs")

submission = pd.read_csv("submission.csv")

print("Submission shape:", submission.shape)
submission.head()

Submission shape: (64062, 4)


,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.002405,NaN,NaN
1,ACCT_000007,0.002405,NaN,NaN
2,ACCT_000009,0.002424,NaN,NaN
3,ACCT_000015,0.002405,NaN,NaN
4,ACCT_000016,0.002405,NaN,NaN


In [2]:
HIGH_RISK_THRESHOLD = 0.25

high_risk_accounts = submission[
    submission["is_mule"] > HIGH_RISK_THRESHOLD
]["account_id"]

print("High risk accounts:", len(high_risk_accounts))

High risk accounts: 1612


In [3]:
# Convert to Python set for fast filtering
high_risk_set = set(high_risk_accounts)

txn_files = sorted(glob("../data/transactions/batch-*/part_*.parquet"))

print("Total transaction files:", len(txn_files))

Total transaction files: 396


In [4]:
txn_list = []

for file in txn_files:
    
    df = (
        pl.scan_parquet(file)
        .filter(pl.col("account_id").is_in(high_risk_set))
        .select([
            "account_id",
            "transaction_timestamp",
            "amount",
            "txn_type"
        ])
        .collect()
    )
    
    if df.height > 0:
        txn_list.append(df)

print("Files containing high-risk activity:", len(txn_list))

Files containing high-risk activity: 387


In [5]:
txns = pl.concat(txn_list)

print("Filtered transaction shape:", txns.shape)

txns = txns.with_columns(
    pl.col("transaction_timestamp").str.to_datetime(
        format="%Y-%m-%dT%H:%M:%S",
        strict=False
    ).alias("ts")
)

txns.head()

Filtered transaction shape: (3281154, 4)


account_id,transaction_timestamp,amount,txn_type,ts
str,str,f64,str,datetime[μs]
"""ACCT_000116""","""2020-07-02T12:20:57""",1000.0,"""C""",2020-07-02 12:20:57
"""ACCT_000116""","""2020-07-03T13:46:20""",469.25,"""C""",2020-07-03 13:46:20
"""ACCT_000116""","""2020-07-06T15:13:18""",181.72,"""C""",2020-07-06 15:13:18
"""ACCT_000116""","""2020-07-07T17:46:51""",3191.42,"""D""",2020-07-07 17:46:51
"""ACCT_000116""","""2020-07-09T09:21:23""",43.8,"""D""",2020-07-09 09:21:23


In [6]:
txns = txns.sort(["account_id", "ts"])

print("Sorted transactions ready.")

Sorted transactions ready.


In [7]:
txns = txns.with_columns(
    (
        pl.col("ts")
        .diff()
        .over("account_id")
        .dt.total_seconds()
    ).alias("gap_seconds")
)

txns.head()

account_id,transaction_timestamp,amount,txn_type,ts,gap_seconds
str,str,f64,str,datetime[μs],i64
"""ACCT_000116""","""2020-07-02T12:20:57""",1000.0,"""C""",2020-07-02 12:20:57,null
"""ACCT_000116""","""2020-07-03T13:46:20""",469.25,"""C""",2020-07-03 13:46:20,91523
"""ACCT_000116""","""2020-07-06T15:13:18""",181.72,"""C""",2020-07-06 15:13:18,264418
"""ACCT_000116""","""2020-07-07T17:46:51""",3191.42,"""D""",2020-07-07 17:46:51,95613
"""ACCT_000116""","""2020-07-09T09:21:23""",43.8,"""D""",2020-07-09 09:21:23,142472


In [188]:
print("Detecting burst transaction windows...")

txns = txns.sort(["account_id","ts"])

txns = txns.with_columns(
    pl.col("ts")
    .diff()
    .over("account_id")
    .dt.total_seconds()
    .alias("gap_seconds")
)

txns = txns.with_columns(
    (pl.col("gap_seconds") > 86400)
    .fill_null(False)
    .cum_sum()
    .over("account_id")
    .alias("burst_group")
)

burst_windows = (
    txns
    .group_by(["account_id","burst_group"])
    .agg([
        pl.len().alias("txn_count"),
        pl.col("ts").min().alias("burst_start"),
        pl.col("ts").max().alias("burst_end")
    ])
)

largest_burst = (
    burst_windows
    .sort(["account_id","txn_count"], descending=[False,True])
    .group_by("account_id")
    .first()
)

largest_burst = largest_burst.select([
    "account_id",
    "burst_start",
    "burst_end"
])

largest_burst.head()

Detecting burst transaction windows...


account_id,burst_start,burst_end
str,datetime[μs],datetime[μs]
"""ACCT_172310""",2025-05-23 14:00:26,2025-05-26 14:14:00
"""ACCT_196313""",2024-09-09 08:08:04,2024-12-14 19:55:25
"""ACCT_110345""",2021-03-14 16:56:29,2021-03-19 17:53:22
"""ACCT_137143""",null,null
"""ACCT_020897""",2025-06-08 01:35:53,2025-06-29 08:38:00


In [189]:
SESSION_GAP = 86400  # 24 hours

txns = txns.with_columns(
    (
        (pl.col("gap_seconds") > SESSION_GAP)
        .fill_null(True)
        .cast(pl.Int32)
        .cum_sum()
        .over("account_id")
    ).alias("session_id")
)

In [190]:
txns.select(["account_id","ts","gap_seconds","session_id"]).head(20)

account_id,ts,gap_seconds,session_id
str,datetime[μs],i64,i32
"""ACCT_000116""",2020-07-02 12:20:57,null,1
"""ACCT_000116""",2020-07-03 13:46:20,91523,2
"""ACCT_000116""",2020-07-06 15:13:18,264418,3
"""ACCT_000116""",2020-07-07 17:46:51,95613,4
"""ACCT_000116""",2020-07-09 09:21:23,142472,5
…,…,…,…
"""ACCT_000116""",2020-07-17 21:44:59,107660,9
"""ACCT_000116""",2020-07-19 07:03:08,119889,10
"""ACCT_000116""",2020-07-20 12:32:22,106154,11


In [191]:
sessions = (
    txns
    .group_by(["account_id", "session_id"])
    .agg([
        pl.col("ts").min().alias("start"),
        pl.col("ts").max().alias("end"),
        pl.len().alias("txn_count"),
        pl.col("amount").sum().alias("total_amount"),
        (pl.col("txn_type") == "C").sum().alias("credits"),
        (pl.col("txn_type") == "D").sum().alias("debits")
    ])
)

sessions.head()

account_id,session_id,start,end,txn_count,total_amount,credits,debits
str,i32,datetime[μs],datetime[μs],u32,f64,u32,u32
"""ACCT_005304""",229,2024-02-25 23:40:39,2024-02-26 18:58:00,9,248892.55,5,4
"""ACCT_045266""",31,2023-05-06 15:41:16,2023-05-06 15:41:16,1,360.26,1,0
"""ACCT_128316""",98,2021-03-29 20:39:09,2021-03-30 14:39:06,3,29831.43,3,0
"""ACCT_003497""",188,2021-12-26 11:32:28,2021-12-26 11:32:28,1,5.54,0,1
"""ACCT_055237""",83,2024-02-11 22:53:59,2024-02-12 16:42:37,4,416.24,2,2


In [192]:
sessions = sessions.with_columns(
(
    pl.col("txn_count") * 0.45 +
    pl.col("total_amount").abs() * 0.00001 +
    pl.col("credits") * 0.25 +
    pl.col("debits") * 0.25 +
    (pl.col("txn_count") > 3).cast(pl.Float64) * 0.5
).alias("suspicion_score")
)

sessions.select(
    ["account_id","session_id","txn_count","total_amount","suspicion_score"]
).head()

account_id,session_id,txn_count,total_amount,suspicion_score
str,i32,u32,f64,f64
"""ACCT_005304""",229,9,248892.55,9.2889255
"""ACCT_045266""",31,1,360.26,0.7036026
"""ACCT_128316""",98,3,29831.43,2.3983143
"""ACCT_003497""",188,1,5.54,0.7000554
"""ACCT_055237""",83,4,416.24,3.3041624


In [193]:
best_windows = (
    sessions
    .sort("suspicion_score", descending=True)
    .group_by("account_id")
    .head(1)
)

best_windows.head()

account_id,session_id,start,end,txn_count,total_amount,credits,debits,suspicion_score
str,i32,datetime[μs],datetime[μs],u32,f64,u32,u32,f64
"""ACCT_043011""",112,2024-11-13 14:01:17,2024-12-09 16:13:19,87,1.2393e6,44,43,73.793379
"""ACCT_070096""",151,2025-05-15 06:28:41,2025-05-23 17:42:26,31,1.2937e6,18,13,35.13745
"""ACCT_105081""",6,2025-01-31 16:58:00,2025-02-02 08:14:00,3,59498.92,0,3,2.6949892
"""ACCT_083620""",111,2025-05-24 11:36:28,2025-06-12 08:43:07,86,4.6409e6,41,45,107.109328
"""ACCT_180168""",100,2025-02-08 10:22:16,2025-03-07 15:33:56,110,1.8996e6,54,56,96.495784


In [194]:
best_pd = best_windows.select([
    "account_id",
    "start",
    "end"
]).to_pandas()

submission = submission.merge(
    best_pd,
    on="account_id",
    how="left"
)

submission["suspicious_start"] = submission["start"]
submission["suspicious_end"] = submission["end"]

submission = submission.drop(columns=["start","end"])

submission.head()

,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.002752,NaT,NaT
1,ACCT_000007,0.002752,NaT,NaT
2,ACCT_000009,0.002773,NaT,NaT
3,ACCT_000015,0.002752,NaT,NaT
4,ACCT_000016,0.002752,NaT,NaT


In [195]:
submission[submission["suspicious_start"].notna()].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.666082,2021-06-06 16:15:02,2021-06-17 07:54:55
92,ACCT_000236,0.884391,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.978161,2025-05-11 16:41:00,2025-05-29 21:22:39
177,ACCT_000484,0.395563,2024-02-12 10:02:06,2024-03-01 15:56:31
231,ACCT_000646,0.884391,2024-05-13 21:31:14,2024-05-17 14:41:56


In [196]:
submission.to_csv("submission_with_temporal_windows.csv", index=False)

print("Final submission saved.")

Final submission saved.


In [197]:
submission["suspicious_start"].notna().sum()

np.int64(1580)

In [198]:
submission[
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].notna())
].head()

,account_id,is_mule,suspicious_start,suspicious_end
92,ACCT_000236,0.884391,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.978161,2025-05-11 16:41:00,2025-05-29 21:22:39
231,ACCT_000646,0.884391,2024-05-13 21:31:14,2024-05-17 14:41:56
309,ACCT_000866,0.782905,2023-10-31 03:40:17,2023-11-02 20:56:23
348,ACCT_000990,0.961122,2024-12-28 12:00:00,2025-01-01 01:16:46


In [199]:
submission[
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].isna())
].head()

,account_id,is_mule,suspicious_start,suspicious_end
1070,ACCT_003281,0.884391,NaT,NaT
1590,ACCT_004974,0.884391,NaT,NaT
7064,ACCT_022184,0.782905,NaT,NaT
10550,ACCT_033209,0.978161,NaT,NaT
11066,ACCT_034824,0.884391,NaT,NaT


In [200]:
txn_basic = pl.read_parquet("../outputs/features_txn_basic.parquet")

fallback_windows = (
    txn_basic
    .select([
        "account_id",
        pl.col("first_txn_date").alias("fallback_start"),
        pl.col("last_txn_date").alias("fallback_end")
    ])
    .to_pandas()
)

submission = submission.merge(
    fallback_windows,
    on="account_id",
    how="left"
)

mask = (
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].isna())
)

submission.loc[mask, "suspicious_start"] = submission.loc[mask, "fallback_start"]
submission.loc[mask, "suspicious_end"] = submission.loc[mask, "fallback_end"]

submission = submission.drop(columns=["fallback_start","fallback_end"])

In [201]:
submission["suspicious_start"].notna().sum()

np.int64(1580)

In [202]:
high_risk = submission[submission["is_mule"] > 0.7]["account_id"]

txn_basic = pl.read_parquet("../outputs/features_txn_basic.parquet")

accounts_with_txns = set(txn_basic["account_id"].to_list())
high_risk_set = set(high_risk)

missing_txn_accounts = high_risk_set - accounts_with_txns

print("High risk accounts:", len(high_risk_set))
print("High risk with transactions:", len(high_risk_set & accounts_with_txns))
print("High risk WITHOUT transactions:", len(missing_txn_accounts))

High risk accounts: 932
High risk with transactions: 932
High risk WITHOUT transactions: 0


In [203]:
submission[submission["suspicious_start"].notna()].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.666082,2021-06-06 16:15:02,2021-06-17 07:54:55
92,ACCT_000236,0.884391,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.978161,2025-05-11 16:41:00,2025-05-29 21:22:39
177,ACCT_000484,0.395563,2024-02-12 10:02:06,2024-03-01 15:56:31
231,ACCT_000646,0.884391,2024-05-13 21:31:14,2024-05-17 14:41:56


In [204]:
best_windows.head()
print(best_windows.shape)

(1612, 9)


In [205]:
submission.columns

Index(['account_id', 'is_mule', 'suspicious_start', 'suspicious_end'], dtype='str')

In [206]:
print(submission["account_id"].dtype)
print(best_windows["account_id"].dtype)

str
String


In [207]:
best_pd = largest_burst.select([
    "account_id",
    "burst_start",
    "burst_end"
]).to_pandas()

print("Applying temporal buffer...")

buffer = pd.Timedelta(hours=6)

best_pd["burst_start"] = best_pd["burst_start"] - buffer
best_pd["burst_end"] = best_pd["burst_end"] + buffer

best_pd["account_id"] = best_pd["account_id"].astype(str)
submission["account_id"] = submission["account_id"].astype(str)

submission = submission.merge(
    best_pd,
    on="account_id",
    how="left"
)

submission["suspicious_start"] = submission["burst_start"]
submission["suspicious_end"] = submission["burst_end"]

print("Removing temporal windows for weak mule probabilities...")

mask = submission["is_mule"] < 0.4

submission.loc[mask, "suspicious_start"] = None
submission.loc[mask, "suspicious_end"] = None
submission = submission.drop(columns=["burst_start","burst_end"])

Applying temporal buffer...
Removing temporal windows for weak mule probabilities...


In [208]:
submission["suspicious_start"].notna().sum()

np.int64(1276)

In [209]:
submission.isnull().sum()

account_id              0
is_mule                 0
suspicious_start    62786
suspicious_end      62786
dtype: int64

In [210]:
submission["suspicious_start"] = submission["suspicious_start"].astype(str).str.replace(" ", "T")
submission["suspicious_end"] = submission["suspicious_end"].astype(str).str.replace(" ", "T")

In [211]:
submission.to_csv("submission_final2.csv", index=False)

In [212]:
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])

print("Max probability:", submission["is_mule"].max())

print("Accounts >0.5:", (submission["is_mule"]>0.5).sum())
print("Accounts >0.7:", (submission["is_mule"]>0.7).sum())

print("Temporal windows:", submission["suspicious_start"].notna().sum())

Rows: 64062
Columns: 4
Max probability: 0.999999
Accounts >0.5: 1271
Accounts >0.7: 932
Temporal windows: 1276


In [213]:
max_p = submission["is_mule"].max()
min_p = submission["is_mule"].min()

submission["is_mule"] = (submission["is_mule"] - min_p) / (max_p - min_p)

In [214]:
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])

print("Max probability:", submission["is_mule"].max())

print("Accounts >0.3:", (submission["is_mule"]>0.3).sum())
print("Accounts >0.5:", (submission["is_mule"]>0.5).sum())
print("Accounts >0.7:", (submission["is_mule"]>0.7).sum())

print("Temporal windows:", submission["suspicious_start"].notna().sum())

Rows: 64062
Columns: 4
Max probability: 1.0
Accounts >0.3: 1611
Accounts >0.5: 1271
Accounts >0.7: 932
Temporal windows: 1276


In [215]:
submission["is_mule"] = submission["is_mule"].clip(1e-6, 1-1e-6)

In [216]:
submission["is_mule"] = submission["is_mule"].astype(float)

In [217]:
submission.to_csv(
    "submission_final2.csv",
    index=False,
    float_format="%.6f"
)

In [218]:
submission["suspicious_start"].notna().sum()

np.int64(1276)

In [219]:
print("FINAL CHECK")

print("Rows:", len(submission))
print("Unique accounts:", submission["account_id"].nunique())
print("Min prob:", submission["is_mule"].min())
print("Max prob:", submission["is_mule"].max())

print("\nThreshold distribution:")
for t in [0.1,0.2,0.3,0.5,0.7]:
    print(t, (submission["is_mule"] > t).sum())

print("\nTemporal windows:", submission["suspicious_start"].notna().sum())
print("Null starts:", submission["suspicious_start"].isna().sum())

FINAL CHECK
Rows: 64062
Unique accounts: 64062
Min prob: 1e-06
Max prob: 0.999999

Threshold distribution:
0.1 2654
0.2 1912
0.3 1611
0.5 1271
0.7 932

Temporal windows: 1276
Null starts: 62786


In [220]:
submission["is_mule"].describe()

count    64062.000000
mean         0.024701
std          0.118418
min          0.000001
25%          0.002751
50%          0.002751
75%          0.002917
max          0.999999
Name: is_mule, dtype: float64

In [221]:
top_1 = submission.sort_values("is_mule", ascending=False).head(1000)
top_5 = submission.sort_values("is_mule", ascending=False).head(5000)

print("Mean prob top 1000:", top_1["is_mule"].mean())
print("Mean prob top 5000:", top_5["is_mule"].mean())
print("Global mean:", submission["is_mule"].mean())

Mean prob top 1000: 0.8576388840656991
Mean prob top 5000: 0.2800229367663203
Global mean: 0.02470129670574535


In [222]:
import numpy as np

for pct in [0.01,0.02,0.05,0.1]:
    k = int(len(submission)*pct)
    mean_prob = submission.sort_values("is_mule", ascending=False).head(k)["is_mule"].mean()
    print(f"Top {int(pct*100)}% mean prob:", mean_prob)

Top 1% mean prob: 0.9120494760561485
Top 2% mean prob: 0.804765012064022
Top 5% mean prob: 0.4266410532146654
Top 10% mean prob: 0.21976241526299062


In [223]:
for t in [0.2,0.3,0.4,0.5]:
    print(t, (submission["is_mule"]>t).sum())

0.2 1912
0.3 1611
0.4 1301
0.5 1271


In [224]:
print(sessions.columns)

['account_id', 'session_id', 'start', 'end', 'txn_count', 'total_amount', 'credits', 'debits', 'suspicion_score']
